# Galactic Star-Forming Regions on a Hybrid Mollweide Map

This notebook draws an all-sky Galactic Mollweide dust map with rectangular zoom panels for crowded star-forming regions and reviewed MALCA dippers. The editable star-forming-region masks live in `malca/data/star_forming_regions.csv`; generated figures are written under `output/notebooks/star_forming_regions/`.

The masks here are approximate visual footprints, not authoritative cloud boundaries.

## Setup

Run this notebook in the `malca` conda environment. It uses `numpy`, `pandas`, `matplotlib`, and `astropy`.

In [ ]:
from pathlib import Path
import sqlite3
import warnings

warnings.filterwarnings("ignore", message="Configuration file not found", module="dustmaps.config")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.colors import LogNorm
from astropy.coordinates import SkyCoord
from astropy.table import Table
import astropy.units as u
from dustmaps.sfd import SFDQuery
from IPython.display import display

from malca.io.notebook_paths import find_repo_root

REPO_ROOT = find_repo_root()
REGION_CSV = REPO_ROOT / "malca" / "data" / "star_forming_regions.csv"
DUSTMAPS_DIR = REPO_ROOT / "data" / "dustmaps"
SFD_DIR = DUSTMAPS_DIR / "sfd"
OUTPUT_DIR = REPO_ROOT / "output" / "notebooks" / "star_forming_regions"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MALCA_REVIEW_DB = REPO_ROOT / "output" / "runs" / "dat3-full-extended_2026-07-01-v4" / "review" / "review.db"

SHOW_DUST_BACKGROUND = True
DOWNLOAD_SFD_IF_MISSING = False
SHOW_MALCA_REVIEW_DIPPERS = True
DUST_GRID_N_L = 2880
DUST_GRID_N_B = 1440

TWO_COLUMN_WIDTH_IN = 7.16
TWO_COLUMN_HEIGHT_IN = 5.35
MAP_AXES_RECT = [0.055, 0.425, 0.815, 0.525]
ORION_ZOOM_RECT = [0.495, 0.135, 0.395, 0.255]
INNER_GALAXY_ZOOM_RECT = [0.065, 0.135, 0.395, 0.255]
COLORBAR_AXES_RECT = [0.885, 0.425, 0.018, 0.525]
LEGEND_ANCHOR = (0.47, 0.012)

BASE_FONT_SIZE_PT = 8.0
AXIS_LABEL_FONT_SIZE_PT = 8.0
TICK_FONT_SIZE_PT = 7.0
REGION_LABEL_FONT_SIZE_PT = 5.0
LEGEND_FONT_SIZE_PT = 6.0
COLORBAR_FONT_SIZE_PT = 7.0
SFR_LABEL_FONT_WEIGHT = "bold"
BLACK_LABEL_FONT_WEIGHT = "normal"
TICK_HALO = [
    pe.Stroke(linewidth=1.75, foreground="white"),
    pe.Stroke(linewidth=0.95, foreground="black"),
    pe.Normal(),
]
LABEL_HALO = [pe.withStroke(linewidth=1.15, foreground="white")]
SFR_TEXT_OUTLINE = [pe.withStroke(linewidth=0.6, foreground="black")]
DIPPER_INSIDE_MARKER_COLOR = "#f2c230"
DIPPER_OUTSIDE_MARKER_COLOR = "#56b4e9"
DIPPER_MARKER_ZORDER = 30


def sfr_text_path_effects(color):
    return [
        pe.withStroke(linewidth=0.6, foreground="black"),
        pe.withStroke(linewidth=0.32, foreground=color),
        pe.Normal(),
    ]


def perceptual_gray_cmap(name="dust_perceptual_gray", lstar_min=0.0, lstar_max=100.0):
    """Neutral grayscale with roughly uniform steps in CIE L* lightness."""
    lstar = np.linspace(lstar_max, lstar_min, 256)
    y = np.where(lstar > 8.0, ((lstar + 16.0) / 116.0) ** 3, lstar / 903.3)
    srgb = np.where(y <= 0.0031308, 12.92 * y, 1.055 * np.power(y, 1.0 / 2.4) - 0.055)
    srgb = np.clip(srgb, 0.0, 1.0)
    colors = np.column_stack([srgb, srgb, srgb, np.ones_like(srgb)])
    return LinearSegmentedColormap.from_list(name, colors)


DUST_CMAP = perceptual_gray_cmap()

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 220,
    "font.size": BASE_FONT_SIZE_PT,
    "font.family": "serif",
    "font.serif": ["cmr10", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "axes.formatter.use_mathtext": True,
    "axes.unicode_minus": False,
})

## Load Editable Region Masks

In [ ]:
REQUIRED_REGION_COLUMNS = [
    "name",
    "l_deg",
    "b_deg",
    "radius_deg",
    "group",
    "distance_pc",
    "notes",
    "source",
]

regions = pd.read_csv(REGION_CSV)
missing = [col for col in REQUIRED_REGION_COLUMNS if col not in regions.columns]
if missing:
    raise ValueError(f"Missing required columns in {REGION_CSV}: {missing}")

regions = regions.copy()
for col in ["l_deg", "b_deg", "radius_deg", "distance_pc"]:
    regions[col] = pd.to_numeric(regions[col], errors="raise")

display(regions)

## Coordinate and Region Geometry Helpers

The plotting helpers wrap Galactic longitude onto `[-180, 180]` degrees. The all-sky panel projects those coordinates onto a Mollweide map centered on `\ell=0`; the zoom panels keep rectangular `\ell,b` axes for local readability.

In [ ]:
def small_circle_points(l_deg, b_deg, radius_deg, n_points=361):
    """Return a spherical small circle around a Galactic center."""
    center = SkyCoord(l=l_deg * u.deg, b=b_deg * u.deg, frame="galactic")
    position_angles = np.linspace(0.0, 360.0, n_points) * u.deg
    circle = center.directional_offset_by(position_angles, radius_deg * u.deg)
    return circle.l.deg, circle.b.deg


def region_center_coords(region_frame):
    return SkyCoord(
        l=region_frame["l_deg"].to_numpy() * u.deg,
        b=region_frame["b_deg"].to_numpy() * u.deg,
        frame="galactic",
    )


def expected_sfd_files(map_dir=SFD_DIR):
    return [map_dir / "SFD_dust_4096_ngp.fits", map_dir / "SFD_dust_4096_sgp.fits"]


def sfd_files_present(map_dir=SFD_DIR):
    return all(path.exists() for path in expected_sfd_files(map_dir))


def fetch_sfd_to_project_cache(map_dir=SFD_DIR):
    """Fetch SFD FITS files into this project without writing ~/.dustmapsrc."""
    from dustmaps import fetch_utils

    map_dir.mkdir(parents=True, exist_ok=True)
    doi = "10.7910/DVN/EWCNL5"
    for pole in ["ngp", "sgp"]:
        filename = f"SFD_dust_4096_{pole}.fits"
        local_path = map_dir / filename
        if local_path.exists():
            continue
        print(f"Downloading {filename} to {local_path}")
        fetch_utils.dataverse_download_doi(
            doi,
            str(local_path),
            file_requirements={"filename": filename},
        )


def load_sfd_query(map_dir=SFD_DIR):
    if not sfd_files_present(map_dir):
        if DOWNLOAD_SFD_IF_MISSING:
            fetch_sfd_to_project_cache(map_dir)
        else:
            expected = "\n".join(str(path) for path in expected_sfd_files(map_dir))
            raise FileNotFoundError(
                "SFD dust files are missing. Set DOWNLOAD_SFD_IF_MISSING = True "
                "and rerun this cell once, or place the files here:\n"
                f"{expected}"
            )
    return SFDQuery(map_dir=str(map_dir))


def make_sfd_lonlat_grid(n_l=DUST_GRID_N_L, n_b=DUST_GRID_N_B):
    """Sample SFD E(B-V) on a regular Galactic longitude/latitude grid."""
    sfd = load_sfd_query()
    x_edges = np.linspace(-180.0, 180.0, n_l + 1)
    y_edges = np.linspace(-90.0, 90.0, n_b + 1)
    x_centers = 0.5 * (x_edges[:-1] + x_edges[1:])
    y_centers = 0.5 * (y_edges[:-1] + y_edges[1:])
    xx, yy = np.meshgrid(x_centers, y_centers)
    l_deg = xx % 360.0
    b_deg = yy
    coords = SkyCoord(l=l_deg.ravel() * u.deg, b=b_deg.ravel() * u.deg, frame="galactic")
    ebv = sfd.query(coords).reshape(n_b, n_l)
    return x_edges, y_edges, ebv


def build_dust_grid():
    if not SHOW_DUST_BACKGROUND:
        return None
    return make_sfd_lonlat_grid()

## Optional Star Catalog Overlay

Set `STAR_CATALOG_PATH` to a CSV or FITS file when you have an additional star table. The loader accepts either Galactic `l/b` columns or ICRS `ra/dec` columns. The synthetic demo is off by default so the main figure shows the reviewed MALCA dippers without extra test points.

In [ ]:
STAR_CATALOG_PATH = None
USE_SYNTHETIC_TEST_STARS = False


def _find_column(table, candidates):
    normalized = {str(col).lower().replace(" ", "_"): col for col in table.columns}
    for candidate in candidates:
        key = candidate.lower().replace(" ", "_")
        if key in normalized:
            return normalized[key]
    return None


def read_star_table(path):
    path = Path(path)
    suffixes = "".join(path.suffixes).lower()
    if suffixes.endswith(".csv"):
        return pd.read_csv(path)
    if suffixes.endswith((".fits", ".fit", ".fits.gz")):
        return Table.read(path).to_pandas()
    raise ValueError(f"Unsupported star table type: {path}")


def star_coords_from_table(star_table):
    l_col = _find_column(star_table, ["l", "l_deg", "gal_l", "gal_l_deg", "glon", "galactic_l"])
    b_col = _find_column(star_table, ["b", "b_deg", "gal_b", "gal_b_deg", "glat", "galactic_b"])
    if l_col is not None and b_col is not None:
        return SkyCoord(
            l=pd.to_numeric(star_table[l_col]).to_numpy() * u.deg,
            b=pd.to_numeric(star_table[b_col]).to_numpy() * u.deg,
            frame="galactic",
        )

    ra_col = _find_column(star_table, ["ra", "ra_deg", "raj2000", "ra_icrs"])
    dec_col = _find_column(star_table, ["dec", "dec_deg", "dej2000", "dec_icrs"])
    if ra_col is not None and dec_col is not None:
        return SkyCoord(
            ra=pd.to_numeric(star_table[ra_col]).to_numpy() * u.deg,
            dec=pd.to_numeric(star_table[dec_col]).to_numpy() * u.deg,
            frame="icrs",
        ).galactic

    raise ValueError(
        "Star table must contain Galactic l/b columns or ICRS ra/dec columns."
    )


def make_synthetic_star_table():
    return pd.DataFrame(
        {
            "source_id": [
                "demo_taurus_inside",
                "demo_orion_overlap",
                "demo_cygnus_inside",
                "demo_inner_galaxy_inside",
                "demo_outside_masks",
            ],
            "l_deg": [170.5, 207.5, 80.5, 352.5, 250.0],
            "b_deg": [-15.5, -16.5, 1.2, 0.8, 55.0],
        }
    )


def load_star_table():
    if STAR_CATALOG_PATH is not None:
        return read_star_table(STAR_CATALOG_PATH), "user catalog"
    if USE_SYNTHETIC_TEST_STARS:
        return make_synthetic_star_table(), "synthetic demo"
    return pd.DataFrame(), "no star catalog"


def assign_region_membership(star_table, regions):
    if star_table.empty:
        return star_table.copy()

    star_coords = star_coords_from_table(star_table)
    sfr_coords = region_center_coords(regions)
    radii = regions["radius_deg"].to_numpy(dtype=float)
    names = regions["name"].to_numpy(dtype=str)

    memberships = []
    min_sep_deg = []
    for star in star_coords:
        separations = star.separation(sfr_coords).deg
        hits = names[separations <= radii]
        memberships.append(";".join(hits))
        min_sep_deg.append(float(np.min(separations)))

    output = star_table.copy()
    output["gal_l_deg"] = star_coords.l.deg
    output["gal_b_deg"] = star_coords.b.deg
    output["sfr_matches"] = memberships
    output["n_sfr_matches"] = [0 if match == "" else match.count(";") + 1 for match in memberships]
    output["nearest_sfr_sep_deg"] = min_sep_deg
    return output


def sqlite_readonly_uri(path):
    return f"file:{Path(path).resolve().as_posix()}?mode=ro&immutable=1"


def load_malca_review_dippers(review_db=MALCA_REVIEW_DB):
    if not SHOW_MALCA_REVIEW_DIPPERS:
        return pd.DataFrame()
    if not Path(review_db).exists():
        print(f"MALCA review DB not found: {review_db}")
        return pd.DataFrame()

    query = """
        SELECT
            c.candidate_id,
            c.ra,
            c.dec,
            c.gal_l AS gal_l_deg,
            c.gal_b AS gal_b_deg,
            c.dipper_score,
            c.dipper_n_dips,
            c.dipper_n_valid_dips,
            r.morphology_primary,
            r.morphology_secondary,
            r.physical_primary,
            r.physical_secondary,
            r.classification_confidence,
            r.status,
            r.updated_at
        FROM reviews AS r
        JOIN candidates AS c USING(candidate_id)
        WHERE r.event_class = 'dipper'
          AND c.gal_l IS NOT NULL
          AND c.gal_b IS NOT NULL
        ORDER BY c.gal_l
    """
    with sqlite3.connect(sqlite_readonly_uri(review_db), uri=True) as conn:
        return pd.read_sql_query(query, conn)


stars_raw, star_source = load_star_table()
stars_with_membership = assign_region_membership(stars_raw, regions)
malca_dippers = load_malca_review_dippers()
malca_dippers_with_membership = assign_region_membership(malca_dippers, regions)

print(f"Star source: {star_source}")
display(stars_with_membership)
print(f"Reviewed MALCA dippers loaded: {len(malca_dippers_with_membership)}")
display(malca_dippers_with_membership.head(10))

## Draw the Map

In [ ]:
def wrap_longitude_deg(l_deg):
    """Wrap Galactic longitude to [-180, 180] for plotting."""
    return ((np.asarray(l_deg, dtype=float) + 180.0) % 360.0) - 180.0


def galactic_to_rectangular(l_deg, b_deg):
    return wrap_longitude_deg(l_deg), np.asarray(b_deg, dtype=float)


def galactic_to_mollweide(l_deg, b_deg):
    l_wrapped = wrap_longitude_deg(l_deg)
    return -np.deg2rad(l_wrapped), np.deg2rad(np.asarray(b_deg, dtype=float))


def split_wrapped_longitude_segments(x, y, max_jump):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    split_at = np.where(np.abs(np.diff(x)) > max_jump)[0] + 1
    return zip(np.split(x, split_at), np.split(y, split_at))


def longitude_tick_label(x_value):
    label = int(round(x_value))
    return rf"${label}^{{\circ}}$"


def ticks_for_xlim(xlim, step):
    left, right = xlim
    lo = min(left, right)
    hi = max(left, right)
    ticks = np.arange(np.floor(lo / step) * step, np.ceil(hi / step) * step + 0.5 * step, step)
    ticks = ticks[(ticks >= lo - 1e-9) & (ticks <= hi + 1e-9)]
    return ticks[::-1] if left > right else ticks


def configure_mollweide_galactic_axis(ax):
    tick_labels = np.arange(180, -181, -30)
    ax.set_xticks(-np.deg2rad(tick_labels))
    ax.set_xticklabels(["" for _ in tick_labels])

    y_ticks = np.arange(-75, 76, 15)
    ax.set_yticks(np.deg2rad(y_ticks))
    ax.set_yticklabels(["" for _ in y_ticks])

    ax.grid(color="0.72", linewidth=0.45, alpha=0.55)
    ax.set_xlabel("")
    ax.set_ylabel(r"$b$ [$^\circ$]", fontsize=AXIS_LABEL_FONT_SIZE_PT, labelpad=2)
    ax.tick_params(axis="both", labelsize=TICK_FONT_SIZE_PT, length=2.6, width=0.45, pad=2)

    for value in tick_labels:
        ax.annotate(
            longitude_tick_label(value),
            xy=(-np.deg2rad(value), 0.0),
            xytext=(0, 0),
            textcoords="offset points",
            ha="center",
            va="center",
            fontsize=TICK_FONT_SIZE_PT,
            color="white",
            path_effects=TICK_HALO,
            annotation_clip=False,
            zorder=40,
        )
    for value in y_ticks:
        if value == 0:
            continue
        ax.annotate(
            rf"${int(value)}^{{\circ}}$",
            xy=(-np.pi, np.deg2rad(value)),
            xytext=(-4, 0),
            textcoords="offset points",
            ha="right",
            va="center",
            fontsize=TICK_FONT_SIZE_PT,
            color="0.05",
            path_effects=LABEL_HALO,
            annotation_clip=False,
            zorder=20,
        )
    for spine in ax.spines.values():
        spine.set_linewidth(0.7)


def configure_rectangular_galactic_axis(
    ax,
    xlim=(180.0, -180.0),
    ylim=(-90.0, 90.0),
    l_step=30.0,
    b_step=15.0,
    show_xlabel=True,
    show_ylabel=True,
    show_y_ticklabels=True,
    y_tick_side="left",
):
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_xticks(ticks_for_xlim(xlim, l_step))
    ax.set_xticklabels([longitude_tick_label(value) for value in ax.get_xticks()])
    y_ticks = np.arange(
        np.ceil(ylim[0] / b_step) * b_step,
        np.floor(ylim[1] / b_step) * b_step + 0.5 * b_step,
        b_step,
    )
    ax.set_yticks(y_ticks)
    ax.set_yticklabels([rf"${int(value)}^{{\circ}}$" for value in y_ticks])
    ax.tick_params(axis="both", labelsize=TICK_FONT_SIZE_PT, length=2.6, width=0.45, pad=2)
    ax.grid(color="0.72", linewidth=0.45, alpha=0.55)
    if show_xlabel:
        ax.set_xlabel(r"$\ell$ [$^\circ$]", fontsize=AXIS_LABEL_FONT_SIZE_PT, labelpad=2)
    else:
        ax.set_xlabel("")
    if show_ylabel:
        ax.set_ylabel(r"$b$ [$^\circ$]", fontsize=AXIS_LABEL_FONT_SIZE_PT, labelpad=2)
    else:
        ax.set_ylabel("")
    if show_y_ticklabels and y_tick_side == "right":
        ax.yaxis.tick_right()
        ax.tick_params(axis="y", labelleft=False, labelright=True)
    elif show_y_ticklabels:
        ax.yaxis.tick_left()
        ax.tick_params(axis="y", labelleft=True, labelright=False)
    else:
        ax.tick_params(axis="y", labelleft=False, labelright=False)
    for spine in ax.spines.values():
        spine.set_linewidth(0.7)


REGION_DISPLAY_NAMES = {
    "lambda Orionis": r"$\lambda$ Ori",
    "Aquila-Serpens": "Aquila/Serpens",
    "Lagoon-Trifid": "Lagoon/Trifid",
    "Eagle-Omega": "Eagle/Omega",
    "W3-W4-W5": "W3/W4/W5",
    "Corona Australis": "CrA",
    "NGC 6334-6357": "NGC 6334/6357",
}

SFR_GROUP_COLORS = {
    "Nearby clouds": "#5f8fc0",
    "Orion complex": "#e89454",
    "Orion-Monoceros": "#81bd7a",
    "Northern clouds": "#c89b92",
    "Rift and inner clouds": "#9b78c6",
    "Massive complexes": "#7f7f7f",
    "Inner galaxy": "#c9c568",
}

MAIN_LABEL_REGIONS = {
    "Taurus",
    "Perseus",
    "Cepheus",
    "Cygnus X",
    "Carina",
    "Vela",
    "Chamaeleon",
    "W3-W4-W5",
}

MAIN_LABEL_PLACEMENTS = {
    "Taurus": (0.00, 1.20),
    "Perseus": (0.00, 1.20),
    "Cepheus": (0.00, 1.20),
    "Cygnus X": (0.00, 1.22),
    "Carina": (0.00, 1.18),
    "Vela": (0.00, 1.18),
    "Chamaeleon": (0.00, 1.18),
    "W3-W4-W5": (0.00, 1.20),
}

ORION_LABEL_PLACEMENTS = {
    "Orion A": (0.00, 1.18),
    "Orion B": (0.00, 1.18),
    "lambda Orionis": (0.00, 1.18),
    "Mon R2": (0.00, 1.20),
    "Rosette": (0.00, 1.22),
}

INNER_LABEL_PLACEMENTS = {
    "Ophiuchus": (0.00, 1.18),
    "Lupus": (0.00, 1.18),
    "Corona Australis": (0.00, 1.20),
    "Aquila-Serpens": (0.00, 1.18),
    "Lagoon-Trifid": (-0.32, 0.72),
    "Eagle-Omega": (0.42, 1.36),
    "NGC 6334-6357": (0.00, 1.20),
}

ORION_ZOOM_XLIM = (-132.0, -174.0)  # l = 228 -> 186 deg, displayed left-to-right.
ORION_ZOOM_YLIM = (-32.0, 5.0)
INNER_GALAXY_ZOOM_XLIM = (42.0, -32.0)  # l = 42 -> 328 deg, crossing l=0.
INNER_GALAXY_ZOOM_YLIM = (-24.0, 30.0)


def dust_norm_from_grid(dust_grid):
    _, _, ebv = dust_grid
    positive = np.asarray(ebv, dtype=float)
    positive = positive[np.isfinite(positive) & (positive > 0.0)]
    if positive.size == 0:
        raise ValueError("Dust grid has no positive finite SFD E(B-V) values to plot.")
    vmin = max(float(np.nanpercentile(positive, 4.0)), 1.0e-3)
    vmax = float(np.nanpercentile(positive, 99.5))
    if vmax <= vmin:
        vmax = vmin * 10.0
    return LogNorm(vmin=vmin, vmax=vmax)


def draw_dust_background(ax, dust_grid, projection="rectangular", norm=None):
    x_edges, y_edges, ebv = dust_grid
    ebv = np.asarray(ebv, dtype=float)
    masked_ebv = np.ma.masked_invalid(np.ma.masked_less_equal(ebv, 0.0))

    if projection == "mollweide":
        x_plot = -np.deg2rad(np.asarray(x_edges)[::-1])
        y_plot = np.deg2rad(y_edges)
        data_plot = masked_ebv[:, ::-1]
    elif projection == "rectangular":
        x_plot = x_edges
        y_plot = y_edges
        data_plot = masked_ebv
    else:
        raise ValueError(f"Unknown projection: {projection}")

    return ax.pcolormesh(
        x_plot,
        y_plot,
        data_plot,
        shading="auto",
        cmap=DUST_CMAP,
        norm=norm if norm is not None else dust_norm_from_grid(dust_grid),
        alpha=1.0,
        zorder=0,
        rasterized=True,
    )


def region_in_view(region, xlim, ylim, margin=2.0):
    x = float(wrap_longitude_deg(region["l_deg"]))
    y = float(region["b_deg"])
    lo_x, hi_x = min(xlim), max(xlim)
    lo_y, hi_y = min(ylim), max(ylim)
    return (lo_x - margin <= x <= hi_x + margin) and (lo_y - margin <= y <= hi_y + margin)


def label_anchor_from_region(region, placements):
    dl_scale, db_scale = placements.get(region["name"], (0.0, 0.0))
    radius = float(region["radius_deg"])
    label_l = float(region["l_deg"]) + dl_scale * radius
    label_b = np.clip(float(region["b_deg"]) + db_scale * radius, -89.0, 89.0)
    return label_l, label_b, float(np.hypot(dl_scale, db_scale))


def projected_coords(l_deg, b_deg, projection):
    if projection == "mollweide":
        return galactic_to_mollweide(l_deg, b_deg)
    if projection == "rectangular":
        return galactic_to_rectangular(l_deg, b_deg)
    raise ValueError(f"Unknown projection: {projection}")


def draw_zoom_footprint_mollweide(ax, xlim, ylim, label):
    x0, x1 = xlim
    y0, y1 = ylim
    n_edge = 120
    bottom_x = np.linspace(x0, x1, n_edge)
    right_y = np.linspace(y0, y1, n_edge)
    top_x = np.linspace(x1, x0, n_edge)
    left_y = np.linspace(y1, y0, n_edge)
    box_x = np.concatenate([bottom_x, np.full(n_edge, x1), top_x, np.full(n_edge, x0)])
    box_y = np.concatenate([np.full(n_edge, y0), right_y, np.full(n_edge, y1), left_y])
    mx, my = galactic_to_mollweide(box_x, box_y)
    for xs, ys in split_wrapped_longitude_segments(mx, my, max_jump=np.pi):
        ax.plot(xs, ys, color="black", linewidth=0.48, alpha=0.78, zorder=6)

    label_x, label_y = galactic_to_mollweide(x0, y1)
    ax.annotate(
        label,
        xy=(label_x, label_y),
        xytext=(2, 2),
        textcoords="offset points",
        ha="left",
        va="bottom",
        fontsize=LEGEND_FONT_SIZE_PT,
        fontweight=BLACK_LABEL_FONT_WEIGHT,
        color="black",
        path_effects=LABEL_HALO,
        zorder=8,
    )


def draw_region_layers(
    ax,
    regions,
    group_colors,
    projection="rectangular",
    label_regions=False,
    label_names=None,
    xlim=None,
    ylim=None,
    label_placements=None,
    clip_labels=False,
):
    if label_placements is None:
        label_placements = {}
    max_jump = np.pi if projection == "mollweide" else 180.0

    for _, region in regions.iterrows():
        circle_l, circle_b = small_circle_points(
            region["l_deg"], region["b_deg"], region["radius_deg"]
        )
        circle_x, circle_y = projected_coords(circle_l, circle_b, projection)
        color = group_colors[str(region["group"])]
        for xs, ys in split_wrapped_longitude_segments(circle_x, circle_y, max_jump=max_jump):
            ax.plot(xs, ys, color=color, linewidth=0.5, alpha=0.88, zorder=3)

        center_x, center_y = projected_coords(region["l_deg"], region["b_deg"], projection)
        ax.scatter(center_x, center_y, s=4.5, color=color, edgecolor="none", linewidth=0, alpha=0.9, zorder=4)

        should_label = label_regions
        if label_names is not None:
            should_label = region["name"] in label_names
        if xlim is not None and ylim is not None:
            should_label = should_label and region_in_view(region, xlim, ylim)
        if not should_label:
            continue

        label_l, label_b, offset_radius = label_anchor_from_region(region, label_placements)
        label_x, label_y = projected_coords(label_l, label_b, projection)
        arrowprops = None
        if offset_radius > 0.35:
            arrowprops = {
                "arrowstyle": "-",
                "color": "0.16",
                "linewidth": 0.32,
                "alpha": 0.45,
                "shrinkA": 0.0,
                "shrinkB": 3.0,
            }

        label_artist = ax.annotate(
            REGION_DISPLAY_NAMES.get(region["name"], region["name"]),
            xy=(center_x, center_y),
            xytext=(label_x, label_y),
            textcoords="data",
            fontsize=REGION_LABEL_FONT_SIZE_PT,
            ha="center",
            va="center",
            color=color,
            fontweight=SFR_LABEL_FONT_WEIGHT,
            path_effects=sfr_text_path_effects(color),
            arrowprops=arrowprops,
            clip_on=clip_labels,
            annotation_clip=clip_labels,
            zorder=5,
        )
        if clip_labels:
            label_artist.set_clip_path(ax.patch)


def draw_star_layers(ax, stars=None, dippers=None, projection="rectangular", dipper_size=24):
    if stars is not None and not stars.empty:
        star_x, star_y = projected_coords(stars["gal_l_deg"], stars["gal_b_deg"], projection)
        in_mask = stars["n_sfr_matches"] > 0
        ax.scatter(star_x[~in_mask], star_y[~in_mask], s=8, color="0.15", alpha=0.35, linewidth=0, zorder=2)
        ax.scatter(star_x[in_mask], star_y[in_mask], s=15, color="black", alpha=0.72, linewidth=0, zorder=6)

    if dippers is not None and not dippers.empty:
        dipper_x, dipper_y = projected_coords(dippers["gal_l_deg"], dippers["gal_b_deg"], projection)
        in_sfr = np.asarray(dippers["n_sfr_matches"].fillna(0) > 0)
        for mask, color in (
            (~in_sfr, DIPPER_OUTSIDE_MARKER_COLOR),
            (in_sfr, DIPPER_INSIDE_MARKER_COLOR),
        ):
            if not np.any(mask):
                continue
            ax.scatter(
                dipper_x[mask],
                dipper_y[mask],
                s=dipper_size,
                marker="*",
                color=color,
                edgecolor="black",
                linewidth=0.32,
                alpha=0.92,
                zorder=DIPPER_MARKER_ZORDER,
            )


def plot_sfr_hybrid_figure(regions, stars=None, dippers=None, dust_grid=None, output_path=None):
    groups = list(dict.fromkeys(regions["group"].astype(str)))
    fallback_cmap = plt.get_cmap("tab20")
    group_colors = {
        group: SFR_GROUP_COLORS.get(group, fallback_cmap(i / max(len(groups), 1)))
        for i, group in enumerate(groups)
    }

    fig = plt.figure(figsize=(TWO_COLUMN_WIDTH_IN, TWO_COLUMN_HEIGHT_IN), constrained_layout=False)
    ax_main = fig.add_axes(MAP_AXES_RECT, projection="mollweide")
    ax_orion = fig.add_axes(ORION_ZOOM_RECT)
    ax_inner = fig.add_axes(INNER_GALAXY_ZOOM_RECT)

    dust_mesh = None
    dust_norm = dust_norm_from_grid(dust_grid) if dust_grid is not None else None
    if dust_grid is not None:
        dust_mesh = draw_dust_background(ax_main, dust_grid, projection="mollweide", norm=dust_norm)
        draw_dust_background(ax_orion, dust_grid, projection="rectangular", norm=dust_norm)
        draw_dust_background(ax_inner, dust_grid, projection="rectangular", norm=dust_norm)

    configure_mollweide_galactic_axis(ax_main)
    configure_rectangular_galactic_axis(
        ax_orion,
        xlim=ORION_ZOOM_XLIM,
        ylim=ORION_ZOOM_YLIM,
        l_step=10.0,
        b_step=10.0,
        show_xlabel=True,
        show_ylabel=False,
        show_y_ticklabels=True,
        y_tick_side="right",
    )
    configure_rectangular_galactic_axis(
        ax_inner,
        xlim=INNER_GALAXY_ZOOM_XLIM,
        ylim=INNER_GALAXY_ZOOM_YLIM,
        l_step=10.0,
        b_step=10.0,
        show_xlabel=True,
        show_ylabel=True,
        show_y_ticklabels=True,
        y_tick_side="left",
    )

    draw_region_layers(
        ax_main,
        regions,
        group_colors,
        projection="mollweide",
        label_names=MAIN_LABEL_REGIONS,
        label_placements=MAIN_LABEL_PLACEMENTS,
    )
    draw_star_layers(ax_main, stars=stars, dippers=dippers, projection="mollweide", dipper_size=20)
    draw_zoom_footprint_mollweide(ax_main, ORION_ZOOM_XLIM, ORION_ZOOM_YLIM, "Orion/Monoceros")
    draw_zoom_footprint_mollweide(ax_main, INNER_GALAXY_ZOOM_XLIM, INNER_GALAXY_ZOOM_YLIM, "Inner Galaxy")

    draw_region_layers(
        ax_orion,
        regions,
        group_colors,
        projection="rectangular",
        label_regions=True,
        xlim=ORION_ZOOM_XLIM,
        ylim=ORION_ZOOM_YLIM,
        label_placements=ORION_LABEL_PLACEMENTS,
        clip_labels=True,
    )
    draw_star_layers(ax_orion, stars=stars, dippers=dippers, projection="rectangular", dipper_size=30)
    ax_orion.text(
        0.02,
        0.96,
        "Orion/Monoceros",
        transform=ax_orion.transAxes,
        ha="left",
        va="top",
        fontsize=LEGEND_FONT_SIZE_PT,
        fontweight=BLACK_LABEL_FONT_WEIGHT,
        path_effects=LABEL_HALO,
        zorder=9,
    )

    draw_region_layers(
        ax_inner,
        regions,
        group_colors,
        projection="rectangular",
        label_regions=True,
        xlim=INNER_GALAXY_ZOOM_XLIM,
        ylim=INNER_GALAXY_ZOOM_YLIM,
        label_placements=INNER_LABEL_PLACEMENTS,
        clip_labels=True,
    )
    draw_star_layers(ax_inner, stars=stars, dippers=dippers, projection="rectangular", dipper_size=30)
    ax_inner.text(
        0.00,
        1.045,
        "Inner Galaxy",
        transform=ax_inner.transAxes,
        ha="left",
        va="bottom",
        fontsize=LEGEND_FONT_SIZE_PT,
        fontweight=BLACK_LABEL_FONT_WEIGHT,
        color="black",
        clip_on=False,
        zorder=9,
    )

    handles = [
        plt.Line2D([0], [0], color=group_colors[group], linewidth=0.8, alpha=0.88, label=group)
        for group in groups
    ]
    if dippers is not None and not dippers.empty:
        n_inside = int((dippers["n_sfr_matches"].fillna(0) > 0).sum())
        n_outside = int(len(dippers) - n_inside)
        for label, color, count in (
            ("dippers in SFRs", DIPPER_INSIDE_MARKER_COLOR, n_inside),
            ("dippers outside", DIPPER_OUTSIDE_MARKER_COLOR, n_outside),
        ):
            if count <= 0:
                continue
            handles.append(
                plt.Line2D(
                    [0], [0], marker="*", color="none", markerfacecolor=color,
                    markeredgecolor="black", markeredgewidth=0.35, linestyle="",
                    markersize=5.2, label=f"{label} ({count})"
                )
            )
    fig.legend(
        handles=handles,
        loc="lower center",
        bbox_to_anchor=LEGEND_ANCHOR,
        ncol=5,
        frameon=False,
        fontsize=LEGEND_FONT_SIZE_PT,
        handlelength=1.5,
        columnspacing=0.85,
        handletextpad=0.42,
        borderaxespad=0.0,
    )
    if dust_mesh is not None:
        cax = fig.add_axes(COLORBAR_AXES_RECT)
        cbar = fig.colorbar(dust_mesh, cax=cax, orientation="vertical")
        cbar.set_label(r"SFD $E(B-V)$ [mag], log stretch", fontsize=COLORBAR_FONT_SIZE_PT, labelpad=4)
        cbar.ax.tick_params(labelsize=TICK_FONT_SIZE_PT, length=2.4, width=0.45, pad=2)
    if output_path is not None:
        fig.savefig(output_path)
    return fig, {"main": ax_main, "orion": ax_orion, "inner": ax_inner}


dust_grid = build_dust_grid()
output_pdf = OUTPUT_DIR / (
    "star_forming_regions_mollweide_dust_zoom.pdf"
    if SHOW_DUST_BACKGROUND
    else "star_forming_regions_mollweide_zoom.pdf"
)
fig, axes = plot_sfr_hybrid_figure(
    regions,
    stars=stars_with_membership,
    dippers=malca_dippers_with_membership,
    dust_grid=dust_grid,
    output_path=output_pdf,
)
plt.show()

print(f"Saved {output_pdf}")


## Membership Summary

In [ ]:
if stars_with_membership.empty:
    print("No star table loaded. Set STAR_CATALOG_PATH above to overlay and classify stars.")
else:
    summary = (
        stars_with_membership.assign(
            sfr_matches=stars_with_membership["sfr_matches"].replace("", "outside all masks")
        )
        .groupby("sfr_matches", dropna=False)
        .size()
        .reset_index(name="n_stars")
        .sort_values("n_stars", ascending=False)
    )
    display(summary)

    multi_match = stars_with_membership[stars_with_membership["n_sfr_matches"] > 1]
    print(f"Stars matching multiple regions: {len(multi_match)}")
    display(multi_match)

if malca_dippers_with_membership.empty:
    print("No reviewed MALCA dippers loaded.")
else:
    dipper_summary = (
        malca_dippers_with_membership.assign(
            sfr_matches=malca_dippers_with_membership["sfr_matches"].replace("", "outside all masks")
        )
        .groupby("sfr_matches", dropna=False)
        .size()
        .reset_index(name="n_reviewed_dippers")
        .sort_values("n_reviewed_dippers", ascending=False)
    )
    display(dipper_summary)

    dipper_multi_match = malca_dippers_with_membership[
        malca_dippers_with_membership["n_sfr_matches"] > 1
    ]
    print(f"Reviewed dippers matching multiple regions: {len(dipper_multi_match)}")
    display(dipper_multi_match)